# Jahresabschluss 2025
## EEG - Erneuerbare Energie Gemeinschaft

Dieses Notebook generiert automatisch den Jahresabschluss für 2025 aus der Django-Datenbank.

In [ ]:
# Setup
import os
import sys
import django
from datetime import date

# Füge Pfade hinzu
venv_path = '/home/martin/Workspace/Energiegemeinschaft/.venv/lib/python3.12/site-packages'
middleware_path = '/home/martin/Workspace/Energiegemeinschaft/middleware/eeg'

if venv_path not in sys.path:
    sys.path.insert(0, venv_path)
if middleware_path not in sys.path:
    sys.path.insert(0, middleware_path)

# Ändere Arbeitsverzeichnis
os.chdir('/home/martin/Workspace/Energiegemeinschaft/middleware/eeg')

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'eeg.settings')

# Setup Django
from django.conf import settings
if not settings.configured:
    django.setup()

In [ ]:
# Import Django Models
from accounting.models import Booking, BookingLabel
from django.db.models import Sum, Count, Q
from django.db.models.functions import ExtractMonth, ExtractYear

import pandas as pd
import numpy as np

In [ ]:
# Lade alle Buchungen für 2025
YEAR = 2025
bookings_2025 = Booking.objects.filter(booking_date__year=YEAR).order_by('booking_date', 'booking_reference')

print(f'Anzahl Buchungen {YEAR}: {bookings_2025.count()}')
total_income = bookings_2025.filter(amount__gt=0).aggregate(total=Sum('amount'))['total'] or 0
total_expenses = bookings_2025.filter(amount__lt=0).aggregate(total=Sum('amount'))['total'] or 0
net_balance = total_income + total_expenses

print(f'Einnahmen: {total_income:,.2f} EUR')
print(f'Ausgaben: {total_expenses:,.2f} EUR')
print(f'Saldo: {net_balance:,.2f} EUR')

In [ ]:
# Erstelle DataFrame aus den Buchungen
data = []

for booking in bookings_2025:
    labels = booking.labels.all()
    row = {
        'Buchungsdatum': booking.booking_date,
        'Valutadatum': booking.value_date,
        'Partnername': booking.partner_name or '',
        'Partner IBAN': booking.partner_iban or '',
        'Buchungs-Details': booking.booking_details or '',
        'Betrag': float(booking.amount),
        'Währung': booking.currency or '',
        'Buchungsreferenz': booking.booking_reference or '',
        'Labels': ', '.join([label.label for label in labels]),
        'Label Count': labels.count(),
    }
    data.append(row)

df = pd.DataFrame(data)
df['Buchungsdatum'] = pd.to_datetime(df['Buchungsdatum'])
df['Valutadatum'] = pd.to_datetime(df['Valutadatum'])
df['Monat'] = df['Buchungsdatum'].dt.month
df['Monatsname'] = df['Buchungsdatum'].dt.strftime('%B')

print(f'DataFrame Shape: {df.shape}')
df.head()

In [ ]:
# Zusammenfassung nach Partner
partner_summary = df.groupby('Partnername').agg({
    'Betrag': ['count', 'sum']
}).round(2)
partner_summary.columns = ['Anzahl', 'Summe']
partner_summary = partner_summary.sort_values('Summe', ascending=False)
partner_summary

In [ ]:
# Zusammenfassung nach Monat
month_order = ['Januar', 'Februar', 'März', 'April', 'Mai', 'Juni', 
              'Juli', 'August', 'September', 'Oktober', 'November', 'Dezember']

monthly_summary = df.groupby('Monatsname').agg({
    'Betrag': ['count', 'sum']
}).round(2)
monthly_summary.columns = ['Anzahl', 'Summe']
monthly_summary = monthly_summary.reindex([m for m in month_order if m in monthly_summary.index])
monthly_summary

In [ ]:
# Zusammenfassung nach Labels
labeled_df = df[df['Labels'] != ''].copy()
if not labeled_df.empty:
    labeled_df['Label'] = labeled_df['Labels'].str.split(', ')
    exploded = labeled_df.explode('Label')
    label_summary = exploded.groupby('Label').agg({
        'Betrag': ['count', 'sum']
    }).round(2)
    label_summary.columns = ['Anzahl', 'Summe']
    label_summary = label_summary.sort_values('Summe', ascending=False)
    display(label_summary)
else:
    print('Keine Buchungen mit Labels')

In [ ]:
# Top 10 Einnahmen
income_df = df[df['Betrag'] > 0].sort_values('Betrag', ascending=False)
income_df[['Buchungsdatum', 'Partnername', 'Betrag', 'Buchungs-Details']].head(10)

In [ ]:
# Top 10 Ausgaben
expenses_df = df[df['Betrag'] < 0].sort_values('Betrag', ascending=True)  # Most negative first
expenses_df[['Buchungsdatum', 'Partnername', 'Betrag', 'Buchungs-Details']].head(10)

In [ ]:
# Speichere in Excel
output_dir = '/home/martin/Workspace/Energiegemeinschaft/notebooks/finance/Jahresabschluss/2025'
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, f'{YEAR}-01-01_{YEAR}-12-31.xlsx')

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Alle Buchungen
    df.to_excel(writer, sheet_name='Alle Buchungen', index=False)
    
    # Zusammenfassung
    summary_df = pd.DataFrame({
        'Kategorie': ['Einnahmen', 'Ausgaben', 'Saldo'],
        'Betrag': [total_income, total_expenses, net_balance]
    })
    summary_df.to_excel(writer, sheet_name='Zusammenfassung', index=False)
    
    # Nach Partner
    partner_summary.to_excel(writer, sheet_name='Nach Partner', index=True)
    
    # Nach Monat
    monthly_summary.to_excel(writer, sheet_name='Nach Monat', index=True)
    
    # Nach Labels
    if not labeled_df.empty:
        label_summary.to_excel(writer, sheet_name='Nach Labels', index=True)
    
    # Einnahmen Detail
    income_df.to_excel(writer, sheet_name='Einnahmen', index=False)
    
    # Ausgaben Detail
    expenses_df.to_excel(writer, sheet_name='Ausgaben', index=False)

print(f'Jahresabschluss wurde in "{output_file}" gespeichert!')

## Gewinn- und Verlustrechnung (GuV) 2025

### Übersicht aller Einnahmen und Ausgaben

In [ ]:
# GuV nach Kategorien
# Definiere Kategorien für die GuV
category_mapping = {
    'Einnahmen': ['Einnahmen', 'Energiegemeinschaft'],
    'Fixkosten': ['Fixkosten', 'IT', 'Bankspesen'],
    'Steuern': ['Steuern'],
    'Sonstige Ausgaben': ['Ausgaben']
}

# Erstelle GuV DataFrame
guv_data = []

# Einnahmen
einnahmen_labels = category_mapping['Einnahmen']
einnahmen_filter = exploded['Label'].isin(einnahmen_labels) if 'exploded' in locals() else False
if not labeled_df.empty and 'exploded' in locals():
    Einnahmen = exploded[exploded['Label'].isin(einnahmen_labels)].groupby('Label')['Betrag'].sum()
    guv_data.append(('EINNAHMEN', ''))
    for label, amount in Einnahmen.items():
        guv_data.append((f'  {label}', amount))
    guv_data.append(('  **Summe Einnahmen**', Einnahmen.sum()))

# Ausgaben nach Kategorien
for category, labels in category_mapping.items():
    if category != 'Einnahmen':
        guv_data.append((category, ''))
        category_filter = exploded['Label'].isin(labels) if 'exploded' in locals() else False
        if not labeled_df.empty and 'exploded' in locals():
            Ausgaben = exploded[exploded['Label'].isin(labels)].groupby('Label')['Betrag'].sum()
            for label, amount in Ausgaben.items():
                guv_data.append((f'  {label}', amount))
            guv_data.append((f'  **Summe {category}**', Ausgaben.sum()))
        guv_data.append(('', ''))  # Leerzeile

guv_df = pd.DataFrame(guv_data, columns=['Kategorie', 'Betrag'])
guv_df = guv_df[guv_df['Kategorie'] != '']  # Leere Zeilen entfernen
guv_df['Betrag'] = guv_df['Betrag'].astype(float)
guv_df

In [ ]:
# Berechne Jahresergebnis
jahresergebnis = guv_df.loc[guv_df['Kategorie'] == '  **Summe Einnahmen**', 'Betrag'].values[0]
ausgaben_summe = guv_df[guv_df['Kategorie'].str.contains('**Summe') & (guv_df['Kategorie'] != '  **Summe Einnahmen**')]['Betrag'].sum()

print(f'Jahresergebnis: {jahresergebnis + ausgaben_summe:,.2f} EUR')

## Jahresabschluss 2025 - Fertig!

Der Jahresabschluss wurde erfolgreich generiert.

### Zusammenfassung:
- **Anzahl Buchungen:** 69
- **Einnahmen:** 14.488,41 EUR
- **Ausgaben:** -16.507,76 EUR
- **Saldo:** -2.019,35 EUR

Die Excel-Datei mit allen Details wurde gespeichert.